# Vigil — Exploratory Data Analysis
**Dataset:** TII-SSRC-23  
**86 sütun** — 83 sayısal feature + `Label` (Benign/Malicious) + `Traffic Type` + `Traffic Subtype`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Kaggle API ile data.csv indir
!pip install kaggle -q

from google.colab import files
print("kaggle.json dosyasını yükle (Kaggle > Settings > API > Create New API Token)")
files.upload()  # kaggle.json — format: {"username":"...","key":"..."}

import os, json, glob

# kaggle.json doğrula
with open('kaggle.json') as f:
    cfg = json.load(f)
assert 'username' in cfg and 'key' in cfg, "kaggle.json hatalı format! Settings > API > Create New API Token"
print(f"Kaggle kullanıcı: {cfg['username']}")

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Dataset indir
!kaggle datasets download -d daniaherzalla/tii-ssrc-23 -p /content/data/ --unzip -q
print("İndirilen dosyalar:", os.listdir('/content/data/'))

# Doğru CSV'yi bul (sample_data hariç)
csv_files = [f for f in glob.glob('/content/data/**/*.csv', recursive=True)
             if 'sample_data' not in f]
assert csv_files, "CSV bulunamadı! İndirme başarısız olmuş olabilir."
PATH = csv_files[0]
print(f"Kullanılacak dosya: {PATH}")

## 1. Genel Bakış

In [ ]:
# RAM'i korumak için önce satır sayısını say, sonra sample yükle
import subprocess
result = subprocess.run(['wc', '-l', PATH], capture_output=True, text=True)
print("Toplam satır:", result.stdout.strip())

# 500k satır EDA için fazlasıyla yeterli (~500MB RAM)
SAMPLE_N = 500_000
df = pd.read_csv(PATH, low_memory=False, on_bad_lines='skip', nrows=SAMPLE_N)
print(f'Shape (sample): {df.shape}')
df.head(3)

In [ ]:
df.info(verbose=False, memory_usage='deep')
print('\nVeri tipleri:')
print(df.dtypes.value_counts())

In [ ]:
df.describe().T.sort_values('std', ascending=False).head(20)

## 2. Eksik & Sonsuz Değerler

In [ ]:
inf_counts = np.isinf(df.select_dtypes(include=np.number)).sum()
null_counts = df.isnull().sum()
problem = pd.DataFrame({'Null': null_counts, 'Inf': inf_counts})
problem = problem[(problem['Null'] > 0) | (problem['Inf'] > 0)]

if problem.empty:
    print('Eksik veya sonsuz değer yok!')
else:
    print(problem)
    problem.plot(kind='bar', figsize=(12, 4), color=['salmon', 'orange'])
    plt.title('Eksik / Sonsuz Değer Sayısı')
    plt.tight_layout()
    plt.savefig('missing_values.png')
    plt.show()

## 3. Sınıf Dağılımı (Label)

In [ ]:
label_counts = df['Label'].value_counts()
print(label_counts)
print(f'\nSınıf dengesi: {label_counts.min()/label_counts.max():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
label_counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='white')
axes[0].set_title('Label Dağılımı (Bar)')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}', (p.get_x()+p.get_width()/2, p.get_height()),
                     ha='center', va='bottom', fontsize=10)

label_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90,
                  colors=['steelblue', 'tomato'])
axes[1].set_title('Label Dağılımı (Pie)')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('label_distribution.png')
plt.show()

## 4. Traffic Type & Subtype Dağılımı

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

tt = df['Traffic Type'].value_counts()
tt.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Traffic Type Dağılımı')
axes[0].set_xlabel('Kayıt Sayısı')

ts = df['Traffic Subtype'].value_counts().head(20)
ts.plot(kind='barh', ax=axes[1], color='teal')
axes[1].set_title('Traffic Subtype (Top 20)')
axes[1].set_xlabel('Kayıt Sayısı')

plt.tight_layout()
plt.savefig('traffic_distribution.png')
plt.show()

## 5. Feature Korelasyon Isı Haritası

In [ ]:
DROP_COLS = ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp', 'Traffic Type', 'Traffic Subtype', 'Label']
num_df = df.drop(columns=[c for c in DROP_COLS if c in df.columns]).select_dtypes(include=np.number)
num_df = num_df.replace([np.inf, -np.inf], np.nan).dropna()

label_num = (df.loc[num_df.index, 'Label'] == 'Malicious').astype(int)
top20 = num_df.corrwith(label_num).abs().sort_values(ascending=False).head(20).index

plt.figure(figsize=(14, 11))
sns.heatmap(num_df[top20].corr(), annot=False, cmap='coolwarm', linewidths=0.3)
plt.title('Feature Korelasyon Isı Haritası (Label ile en yüksek 20 feature)')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.show()

## 6. Label ile En Çok Korelasyonlu Feature'lar

In [ ]:
corr_with_label = num_df.corrwith(label_num).abs().sort_values(ascending=False).head(20)

plt.figure(figsize=(10, 6))
corr_with_label.plot(kind='barh', color='darkorange')
plt.title('Label ile Korelasyon (Mutlak Değer, Top 20)')
plt.xlabel('Korelasyon')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_label_correlation.png')
plt.show()

print(corr_with_label)

## 7. Benign vs Malicious — Feature Dağılımı

In [ ]:
top5 = corr_with_label.head(5).index.tolist()
plot_df = num_df[top5].copy()
plot_df['Label'] = df.loc[num_df.index, 'Label']

fig, axes = plt.subplots(1, 5, figsize=(20, 5))
for i, col in enumerate(top5):
    plot_df.boxplot(column=col, by='Label', ax=axes[i])
    axes[i].set_title(col[:20])
    axes[i].set_xlabel('')
plt.suptitle('Benign vs Malicious — Top 5 Feature')
plt.tight_layout()
plt.savefig('boxplots.png')
plt.show()

## 8. Protocol Dağılımı

In [ ]:
proto = df['Protocol'].value_counts()
plt.figure(figsize=(7, 4))
proto.plot(kind='bar', color='mediumpurple')
plt.title('Protocol Dağılımı')
plt.xlabel('Protocol')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('protocol_distribution.png')
plt.show()
print(proto)

## 9. Görselleri İndir

In [ ]:
import os
pngs = [f for f in os.listdir('.') if f.endswith('.png')]
print('İndirilecek dosyalar:', pngs)
for f in pngs:
    files.download(f)